In [1]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Load the MNIST dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 41s 4us/step


In [2]:
# Normalize pixel values to be between 0 and 1
x_train = x_train / 255.0
x_test = x_test / 255.0

# Reshape the data for the CNN (add a channel dimension)
# The shape will go from (60000, 28, 28) to (60000, 28, 28, 1)
x_train_cnn = x_train.reshape(x_train.shape[0], 28, 28, 1)
x_test_cnn = x_test.reshape(x_test.shape[0], 28, 28, 1)

print("Shape for ANN:", x_train.shape)
print("Shape for CNN:", x_train_cnn.shape)

Shape for ANN: (60000, 28, 28)
Shape for CNN: (60000, 28, 28, 1)


In [3]:
# Create the ANN model
ann_model = Sequential([
    Flatten(input_shape=(28, 28)),  # Flattens the 28x28 image into a 1D array of 784 pixels
    Dense(128, activation='relu'),   # A fully connected layer with 128 neurons
    Dense(10, activation='softmax')  # The output layer with 10 neurons (for digits 0-9)
])

# Print the model summary to see the architecture
print("ANN Architecture:")
ann_model.summary()

C:\Users\LAPTOP CLINIC\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


ANN Architecture:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ flatten (Flatten)                    │ (None, 784)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │         100,480 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 10)                  │           1,290 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 101,770 (397.54 KB)

 Trainable params: 101,770 (397.54 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
# Create the CNN model
cnn_model = Sequential([
    # Convolutional layer: 32 filters of size 3x3
    Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    # Pooling layer: takes the max value from a 2x2 grid
    MaxPooling2D((2, 2)),
    # Flatten the 2D feature maps into a 1D vector
    Flatten(),
    # A fully connected layer for classification
    Dense(128, activation='relu'),
    # The output layer
    Dense(10, activation='softmax')
])

# Print the model summary
print("\nCNN Architecture:")
cnn_model.summary()


CNN Architecture:


C:\Users\LAPTOP CLINIC\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 26, 26, 32)          │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 13, 13, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_1 (Flatten)                  │ (None, 5408)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 128)                 │         692,352 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 10)                  │           1,290 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 693,962 (2.65 MB)

 Trainable params: 693,962 (2.65 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# Callbacks for the ANN model
ann_callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, verbose=1),
    ModelCheckpoint('best_ann_model.keras', save_best_only=True, monitor='val_accuracy', mode='max')
]

# Callbacks for the CNN model
cnn_callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, verbose=1),
    ModelCheckpoint('best_cnn_model.keras', save_best_only=True, monitor='val_accuracy', mode='max')
]

# Common training parameters
EPOCHS = 10
BATCH_SIZE = 32

In [7]:
# Compile the ANN model
ann_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

# Train the ANN model
print("\n--- Training ANN ---")
ann_history = ann_model.fit(x_train, y_train,
                            epochs=EPOCHS,
                            batch_size=BATCH_SIZE,
                            validation_split=0.1, # Use 10% of training data for validation
                            callbacks=ann_callbacks)


--- Training ANN ---
Epoch 1/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9922 - loss: 0.0243 - val_accuracy: 0.9785 - val_loss: 0.0762
Epoch 2/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9957 - loss: 0.0165 - val_accuracy: 0.9795 - val_loss: 0.0809
Epoch 3/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9958 - loss: 0.0136 - val_accuracy: 0.9805 - val_loss: 0.0828
Epoch 4/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9963 - loss: 0.0134 - val_accuracy: 0.9787 - val_loss: 0.0818
Epoch 4: early stopping


In [8]:
# Compile the CNN model
cnn_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

# Train the CNN model (use the reshaped data)
print("\n--- Training CNN ---")
cnn_history = cnn_model.fit(x_train_cnn, y_train,
                            epochs=EPOCHS,
                            batch_size=BATCH_SIZE,
                            validation_split=0.1,
                            callbacks=cnn_callbacks)


--- Training CNN ---
Epoch 1/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 36s 20ms/step - accuracy: 0.9053 - loss: 0.3250 - val_accuracy: 0.9820 - val_loss: 0.0626
Epoch 2/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 35s 21ms/step - accuracy: 0.9824 - loss: 0.0591 - val_accuracy: 0.9853 - val_loss: 0.0577
Epoch 3/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 36s 21ms/step - accuracy: 0.9895 - loss: 0.0344 - val_accuracy: 0.9857 - val_loss: 0.0518
Epoch 4/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 37s 22ms/step - accuracy: 0.9930 - loss: 0.0225 - val_accuracy: 0.9880 - val_loss: 0.0446
Epoch 5/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 38s 22ms/step - accuracy: 0.9955 - loss: 0.0147 - val_accuracy: 0.9867 - val_loss: 0.0498
Epoch 6/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 36s 21ms/step - accuracy: 0.9969 - loss: 0.0105 - val_accuracy: 0.9862 - val_loss: 0.0556
Epoch 7/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 38s 23ms/step - accuracy: 0.9972 - loss: 0.0074 - val_accuracy: 0.9848 - val_loss: 0.0643
Epoch 7: early stopping


In [9]:
# Load the best models
best_ann = tf.keras.models.load_model('best_ann_model.keras')
best_cnn = tf.keras.models.load_model('best_cnn_model.keras')

# Evaluate the ANN
print("\n--- Evaluating ANN on Test Data ---")
ann_loss, ann_accuracy = best_ann.evaluate(x_test, y_test)
print(f"ANN Test Accuracy: {ann_accuracy * 100:.2f}%")

# Evaluate the CNN
print("\n--- Evaluating CNN on Test Data ---")
cnn_loss, cnn_accuracy = best_cnn.evaluate(x_test_cnn, y_test)
print(f"CNN Test Accuracy: {cnn_accuracy * 100:.2f}%")


--- Evaluating ANN on Test Data ---
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9754 - loss: 0.0905
ANN Test Accuracy: 97.74%

--- Evaluating CNN on Test Data ---
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9837 - loss: 0.0479
CNN Test Accuracy: 98.64%
